In [26]:
import pandas as pd
import numpy as np
import time

In [27]:
file_path = r'chain_assignment.csv'

df= pd.read_csv(file_path, encoding='utf-8')

df

,用户ID,姓名,性别,年龄,城市,消费金额(元),会员等级,是否达标
0,1001,张三,男,25,北京,2999,VIP3,否
1,1002,李四,女,32,上海,199,VIP2,否
2,1003,王五,男,28,广州,318,VIP1,否
3,1004,赵六,女,40,深圳,99,VIP3,否
4,1005,钱七,男,35,北京,2999,VIP2,是
5,1006,孙八,女,29,上海,299,VIP1,否
6,1007,周九,男,45,广州,129,VIP3,否
7,1008,吴十,女,27,深圳,799,VIP2,是
8,1009,郑十一,男,33,北京,599,VIP1,否
9,1010,冯十二,女,38,上海,1999,VIP3,是


In [28]:
df = pd.concat([df]*1000, ignore_index=True)
print(f"测试数据量：{len(df)} 行")

测试数据量：10000 行


In [29]:
df

,用户ID,姓名,性别,年龄,城市,消费金额(元),会员等级,是否达标
0,1001,张三,男,25,北京,2999,VIP3,否
1,1002,李四,女,32,上海,199,VIP2,否
2,1003,王五,男,28,广州,318,VIP1,否
3,1004,赵六,女,40,深圳,99,VIP3,否
4,1005,钱七,男,35,北京,2999,VIP2,是
...,...,...,...,...,...,...,...,...
9995,1006,孙八,女,29,上海,299,VIP1,否
9996,1007,周九,男,45,广州,129,VIP3,否
9997,1008,吴十,女,27,深圳,799,VIP2,是
9998,1009,郑十一,男,33,北京,599,VIP1,否


In [30]:
chain_count_before = len(df[(df['城市']=='北京') & (df['会员等级']=='VIP3') & (df['消费金额(元)']>2000) & (df['是否达标']=='否')])
print(chain_count_before)
start_time = time.time()
# 链式赋值（df[]][...]）
df[df['城市'] == '北京'][df['会员等级'] == 'VIP3'][df['消费金额(元)'] > 2000]['是否达标'] = '是'
chain_time = time.time() - start_time

# 验证修改结果（大概率未生效）
chain_count = len(df[(df['城市']=='北京') & (df['会员等级']=='VIP3') & (df['消费金额(元)']>2000) & (df['是否达标']=='是')])
print("\n1. 链式赋值（错误方式）：")
print(f"耗时：{chain_time:.4f} 秒")
print(f"符合条件且修改成功的记录数：{chain_count}（大概率为0，修改失效）")
print("⚠️  控制台会触发SettingWithCopyWarning警告！")

1000

1. 链式赋值（错误方式）：
耗时：0.0039 秒
符合条件且修改成功的记录数：0（大概率为0，修改失效）
⚠️  控制台会触发SettingWithCopyWarning警告！


C:\Users\LocalHost\AppData\Local\Temp\ipykernel_31760\2999544229.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[df['城市'] == '北京'][df['会员等级'] == 'VIP3'][df['消费金额(元)'] > 2000]['是否达标'] = '是'
C:\Users\LocalHost\AppData\Local\Temp\ipykernel_31760\2999544229.py:5: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[df['城市'] == '北京'][df['会员等级'] == 'VIP3'][df['消费金额(元)'] > 2000]['是否达标'] = '是'
C:\Users\LocalHost\AppData\Local\Temp\ipykernel_31760\2999544229.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pand

In [31]:
# 重置数据
df = pd.read_csv(file_path, encoding='utf-8')
df = pd.concat([df]*1000, ignore_index=True)

start_time = time.time()
# loc单步赋值：df.loc[条件, 列名] = 值（原子操作，无警告，修改生效）
df.loc[(df['城市'] == '北京') & (df['会员等级'] == 'VIP3') & (df['消费金额(元)'] > 2000), '是否达标'] = '是'
loc_time = time.time() - start_time

# 验证修改结果
loc_count = len(df[(df['城市']=='北京') & (df['会员等级']=='VIP3') & (df['消费金额(元)']>2000) & (df['是否达标']=='是')])
print("\n2. loc单步赋值（推荐方式）：")
print(f"耗时：{loc_time:.4f} 秒（比链式赋值快 {(chain_time-loc_time)/chain_time*100:.2f}%）")
print(f"符合条件且修改成功的记录数：{loc_count}（修改生效）")


2. loc单步赋值（推荐方式）：
耗时：0.0058 秒（比链式赋值快 -48.65%）
符合条件且修改成功的记录数：1000（修改生效）


In [32]:
# 重置数据
df = pd.read_csv(file_path, encoding='utf-8')
df = pd.concat([df]*1000, ignore_index=True)

start_time = time.time()
# 步骤1：定义复杂条件（可读性更高）
condition = (df['城市'].isin(['北京', '上海'])) & \
            (df['会员等级'] == 'VIP2') & \
            (df['消费金额(元)'] > 500)
# 步骤2：loc赋值
df.loc[condition, '是否达标'] = '是'
cond_time = time.time() - start_time

# 验证
cond_count = len(df[condition & (df['是否达标']=='是')])
print("\n3. 条件变量+loc赋值（复杂场景推荐）：")
print(f"耗时：{cond_time:.4f} 秒")
print(f"符合条件且修改成功的记录数：{cond_count}")


3. 条件变量+loc赋值（复杂场景推荐）：
耗时：0.0062 秒
符合条件且修改成功的记录数：1000


In [33]:
# 重置数据
df = pd.read_csv(file_path, encoding='utf-8')
df = pd.concat([df]*1000, ignore_index=True)

start_time = time.time()
# 多列批量赋值（仍用loc单步操作）
condition_multi = df['年龄'] > 35
df.loc[condition_multi, ['是否达标', '会员等级']] = ['是', 'VIP4']  # 多列同时修改
multi_time = time.time() - start_time

# 验证
multi_count = len(df[condition_multi & (df['会员等级']=='VIP4')])
print("\n4. loc多列批量赋值：")
print(f"耗时：{multi_time:.4f} 秒")
print(f"年龄>35且会员等级改为VIP4的记录数：{multi_count}")


4. loc多列批量赋值：
耗时：0.0111 秒
年龄>35且会员等级改为VIP4的记录数：3000


In [34]:
df = pd.read_csv(file_path, encoding='utf-8')
df = pd.concat([df]*1000, ignore_index=True)
print("\n5. 链式赋值坑点解析：")
# 链式赋值 df[A][B][C] = 值 → 实际是 df[A] 返回副本/视图 → 后续修改仅作用于副本，原数据不变
# loc是原子操作 → 直接修改原数据，无中间副本，既生效又高效
# 错误示例：先切片再修改（等同于链式赋值）
df_slice = df[df['城市'] == '广州']  # 返回副本
df_slice['是否达标'] = '是'  # 修改副本，原df不变
origin_count = len(df[df['城市']=='广州'][df['是否达标']=='是'])
print(f"切片后修改副本，原数据是否达标数：{origin_count}（仍为0，修改仅在副本生效）")
# 正确示例：切片时用loc（返回视图，修改生效）
df.loc[df['城市'] == '深圳', '是否达标'] = '是'
origin_sz_count = len(df[df['城市']=='深圳'][df['是否达标']=='是'])
print(f"loc修改深圳数据，原数据是否达标数：{origin_sz_count}（修改生效）")


5. 链式赋值坑点解析：
切片后修改副本，原数据是否达标数：0（仍为0，修改仅在副本生效）
loc修改深圳数据，原数据是否达标数：2000（修改生效）


C:\Users\LocalHost\AppData\Local\Temp\ipykernel_31760\3098888983.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  origin_count = len(df[df['城市']=='广州'][df['是否达标']=='是'])
C:\Users\LocalHost\AppData\Local\Temp\ipykernel_31760\3098888983.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  origin_sz_count = len(df[df['城市']=='深圳'][df['是否达标']=='是'])


In [35]:
summary = pd.DataFrame(
    {
        '赋值方式': ['链式赋值', 'loc单步赋值', '条件变量+loc', 'loc多列赋值'],
        '耗时(秒)': [chain_time, loc_time, cond_time, multi_time],
        '修改是否生效': ['否', '是', '是', '是'],
        '是否触发警告': ['是', '否', '否', '否']
    }
).round(4)
print("\n6. 优化效果汇总：")
print(summary)


6. 优化效果汇总：
       赋值方式   耗时(秒) 修改是否生效 是否触发警告
0      链式赋值  0.0039      否      是
1   loc单步赋值  0.0058      是      否
2  条件变量+loc  0.0062      是      否
3   loc多列赋值  0.0111      是      否
